# Experiment 8: Time Series Forecasting and Trend Lines

**Aim:** Calculate a moving average, fit a linear trend line and forecast the next three months.

**Dataset limitation:** The supplied orders cover January–March 2026 with only 15 order records. The forecast is a small educational demonstration and should not be treated as a reliable business forecast.

## Import libraries and load the supplied CSV files
The five dimension/fact tables are joined using their ID/date keys. Sales, cost and profit are calculated from the supplied unit price, unit cost, units sold and discount.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

DATA_DIR = Path("datasets")

def load_data():
    required_files = [
        "Dim_Customer.csv", "Dim_Date.csv", "Dim_Geography.csv",
        "Dim_Product.csv", "Fact_Orders.csv"
    ]
    missing = [f for f in required_files if not (DATA_DIR / f).exists()]
    if missing:
        raise FileNotFoundError(
            "Missing CSV file(s): " + ", ".join(missing) +
            ". Put all five CSV files in the datasets folder beside this notebook."
        )
    customer = pd.read_csv(DATA_DIR / "Dim_Customer.csv")
    dates = pd.read_csv(DATA_DIR / "Dim_Date.csv")
    geography = pd.read_csv(DATA_DIR / "Dim_Geography.csv")
    product = pd.read_csv(DATA_DIR / "Dim_Product.csv")
    orders = pd.read_csv(DATA_DIR / "Fact_Orders.csv")

    # Join the fact table with dimension tables using their matching keys.
    data = orders.merge(customer, on="Customer_ID", how="left")
    data = data.merge(product, on="Product_ID", how="left")
    data = data.merge(geography, on="Geo_ID", how="left")
    data = data.merge(dates, left_on="Order_Date", right_on="Date", how="left")

    data["Order_Date"] = pd.to_datetime(data["Order_Date"], errors="coerce")
    data["Ship_Date"] = pd.to_datetime(data["Ship_Date"], errors="coerce")
    data["Sales"] = data["Units_Sold"] * data["Unit_Price"] * (1 - data["Discount_Pct"])
    data["Cost"] = data["Units_Sold"] * data["Unit_Cost"]
    data["Profit"] = data["Sales"] - data["Cost"]
    data["Profit_Margin"] = np.where(data["Sales"] != 0,
                                     data["Profit"] / data["Sales"] * 100, 0)
    data["Order_Month"] = data["Order_Date"].dt.to_period("M").astype(str)
    return data

## Python code and visualization

In [ ]:
from sklearn.linear_model import LinearRegression

df = load_data()
monthly = df.groupby("Order_Month", as_index=False)["Sales"].sum().sort_values("Order_Month")
monthly["Month_Number"] = np.arange(len(monthly))
monthly["Moving_Average"] = monthly["Sales"].rolling(window=2, min_periods=1).mean()

# Fit a simple linear regression trend.
X = monthly[["Month_Number"]]
y = monthly["Sales"]
model = LinearRegression()
model.fit(X, y)
monthly["Trend"] = model.predict(X)

# Forecast the next 3 months.
future_x = np.arange(len(monthly), len(monthly) + 3).reshape(-1, 1)
future_sales = model.predict(future_x)
last_month = pd.Period(monthly["Order_Month"].iloc[-1], freq="M")
future_months = [(last_month + i).strftime("%Y-%m") for i in range(1, 4)]
forecast = pd.DataFrame({"Order_Month": future_months, "Forecast_Sales": future_sales})

plt.figure(figsize=(10, 6))
plt.plot(monthly["Order_Month"], monthly["Sales"], marker="o", label="Actual Sales")
plt.plot(monthly["Order_Month"], monthly["Moving_Average"], marker="o",
         label="2-month Moving Average")
plt.plot(monthly["Order_Month"], monthly["Trend"], linestyle="--", label="Trend Line")
plt.plot(forecast["Order_Month"], forecast["Forecast_Sales"],
         marker="o", linestyle=":", label="3-month Forecast")
plt.title("Monthly Sales Trend and Forecast")
plt.xlabel("Month")
plt.ylabel("Sales (₹)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

display(forecast)

## Result
The experiment was performed using the supplied retail dataset. Run all cells to generate the tables, calculations and visualizations.